# Feature Scaling for House-Price Prediction

This notebook uses public Ames Housing data to show why feature scaling matters for gradient-based linear regression.

## Learning goals

- Inspect features with very different numerical ranges.
- Standardize features with `StandardScaler`.
- Use a scikit-learn `Pipeline` to prevent data leakage.
- Compare gradient-descent training with and without feature scaling.

In [ ]:
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import SGDRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error, r2_score

## Load the public Ames Housing data

We will predict sale price using three features: above-ground living area, lot area, and year built. Their numerical ranges are very different: lot area can be in the tens of thousands of ft², while year built is close to 2,000.

Source: [OpenML house_prices dataset](https://www.openml.org/d/42165).

In [ ]:
ames = fetch_openml(data_id=42165, as_frame=True, parser='auto')
features = ['GrLivArea', 'LotArea', 'YearBuilt']
housing = ames.frame[features + ['SalePrice']].dropna().copy()
housing = housing[(housing['GrLivArea'] < 3_000) & (housing['SalePrice'].astype(float) < 500_000)]
X = housing[features].astype(float)
y = housing['SalePrice'].astype(float)
X.describe().loc[['mean', 'std', 'min', 'max']]

## Why scale features?

Gradient descent changes one coefficient at a time. If one feature has values around 20,000 while another has values around 2,000, the first feature can dominate the updates. A learning rate that is stable for one feature may be far too large or too small for another.

Standardization transforms each feature to approximately zero mean and unit standard deviation: $z = (x - u) / igma$. It changes the representation used during fitting, not the underlying information.

In [ ]:
X.plot(kind='box', figsize=(9, 4), grid=True)
plt.title('Original feature scales')
plt.ylabel('Original units')
plt.show()

scaler_preview = StandardScaler()
X_scaled_preview = scaler_preview.fit_transform(X)
plt.boxplot(X_scaled_preview, tick_labels=features)
plt.title('Feature scales after standardization')
plt.ylabel('Standardized value')
plt.grid(alpha=0.25)
plt.show()

## Split before scaling

Fit the scaler only on training data. If the mean and standard deviation from the test set influence scaling, information from the test set has leaked into training. A `Pipeline` enforces the correct sequence automatically.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

## Fit a scaled gradient-descent model

`SGDRegressor` uses stochastic gradient descent. The pipeline first standardizes the input features and then fits the regression model. `TransformedTargetRegressor` also standardizes the target during fitting, then converts predictions back to dollars.

In [ ]:
scaled_sgd = TransformedTargetRegressor(
    regressor=make_pipeline(
        StandardScaler(),
        SGDRegressor(max_iter=5_000, tol=1e-5, eta0=0.01, learning_rate='constant', random_state=42)
    ),
    transformer=StandardScaler()
)
scaled_sgd.fit(X_train, y_train)

scaled_predictions = scaled_sgd.predict(X_test)
scaled_rmse = mean_squared_error(y_test, scaled_predictions) ** 0.5
scaled_r2 = r2_score(y_test, scaled_predictions)
print(f'Scaled SGD test RMSE: ${scaled_rmse:,.0f}')
print(f'Scaled SGD test R²: {scaled_r2:.3f}')

## Compare with an unscaled model

The unscaled model needs a much smaller learning rate because `LotArea` has large values. This comparison is intentionally conservative: the tiny learning rate avoids unstable updates, but makes learning slower.

In [ ]:
unscaled_sgd = TransformedTargetRegressor(
    regressor=SGDRegressor(
        max_iter=5_000, tol=1e-5, eta0=0.000001, learning_rate='constant', random_state=42
    ),
    transformer=StandardScaler()
)
unscaled_sgd.fit(X_train, y_train)

unscaled_predictions = unscaled_sgd.predict(X_test)
unscaled_rmse = mean_squared_error(y_test, unscaled_predictions) ** 0.5
print(f'Unscaled SGD test RMSE: ${unscaled_rmse:,.0f}')
print(f'Scaled SGD test RMSE:   ${scaled_rmse:,.0f}')

## Experiment

1. Change the scaled model's `eta0` from `0.01` to `0.1` and then to `0.001`.
2. Remove `StandardScaler()` from the pipeline and rerun the model with `eta0=0.01`.
3. Compare the test RMSE values. Which version trains reliably and why?

Feature scaling is particularly important for gradient descent, regularized linear models, nearest-neighbor methods, support-vector machines, and neural networks. Ordinary least squares with `LinearRegression` can usually fit without it, but scaling still makes coefficients and related workflows easier to handle.

## Summary

Standardization places input features on comparable numerical scales. Always split into training and test data first, then fit the scaler within a pipeline using the training data only. This avoids data leakage and makes gradient-based optimization easier to tune.